In [1]:
from sentence_transformers import SentenceTransformer

from transformers import pipeline

import faiss
import numpy as np

from tools import *

from memory import *

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


In [3]:
documents = []

documents.append(
    get_client_profile(
        "Acme Corp"
    )
)

documents.append(
    get_meeting_notes(
        "Acme Corp"
    )
)

documents.append(
    get_action_items(
        "Acme Corp"
    )
)

print(
    "Documents Loaded:",
    len(documents)
)

Documents Loaded: 3


In [4]:
embedding_model = (
    SentenceTransformer(
        "all-MiniLM-L6-v2"
    )
)

/home/nineleaps/.local/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
embeddings = (
    embedding_model.encode(
        documents
    )
)

embeddings = (
    embeddings.astype(
        "float32"
    )
)

print(
    embeddings.shape
)

(3, 384)


In [6]:
faiss.normalize_L2(
    embeddings
)

index = (
    faiss.IndexFlatL2(
        384
    )
)

index.add(
    embeddings
)

print(
    "Vectors:",
    index.ntotal
)

Vectors: 3


In [15]:
llm = pipeline(
    "text2text-generation",
    model="google/flan-t5-large"
)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [8]:
def retrieve_context(
    query
):

    query_embedding = (
        embedding_model.encode(
            [query]
        )
    )

    query_embedding = (
        query_embedding.astype(
            "float32"
        )
    )

    faiss.normalize_L2(
        query_embedding
    )

    distances, indices = (
        index.search(
            query_embedding,
            3
        )
    )

    retrieved_docs = []

    for idx in indices[0]:

        retrieved_docs.append(
            documents[idx]
        )

    return "\n".join(
        retrieved_docs
    )

In [9]:
def agent_plan(
    user_query
):

    plan = []

    if "meeting" in user_query.lower():

        plan.append(
            "client_profile"
        )

        plan.append(
            "meeting_notes"
        )

        plan.append(
            "memory"
        )

        plan.append(
            "action_items"
        )

    return plan

In [10]:
def execute_plan(
    client_name,
    plan
):

    results = {}

    if "client_profile" in plan:

        results[
            "profile"
        ] = get_client_profile(
            client_name
        )

    if "meeting_notes" in plan:

        results[
            "notes"
        ] = get_meeting_notes(
            client_name
        )

    if "memory" in plan:

        results[
            "memory"
        ] = get_long_term_memory(
            client_name
        )

    if "action_items" in plan:

        results[
            "actions"
        ] = get_action_items(
            client_name
        )

    return results

In [21]:
def generate_meeting_brief(gathered_info):

    brief = f"""
    MEETING BRIEF
    ==================================================
    
    CLIENT PROFILE
    --------------------------------------------------
    {gathered_info['profile']}
    
    PREVIOUS MEETING NOTES
    --------------------------------------------------
    {gathered_info['notes']}
    
    LONG TERM MEMORY
    --------------------------------------------------
    {gathered_info['memory']}
    
    OPEN ACTION ITEMS
    --------------------------------------------------
    {gathered_info['actions']}
    
    SUGGESTED NEXT STEPS
    --------------------------------------------------
    1. Discuss predictive maintenance requirements.
    2. Present analytics dashboard demo.
    3. Review onboarding timeline.
    4. Confirm pricing proposal.
    5. Schedule next follow-up meeting.
    
    """

    return brief

In [22]:
while True:

    user_query = input(
        "\nYou: "
    )

    if user_query.lower() == "exit":

        print(
            "Agent: Goodbye!"
        )

        break

    add_message(
        "user",
        user_query
    )

    plan = agent_plan(
        user_query
    )

    print(
        "\nAgent Plan:"
    )

    print(plan)

    gathered_info = (
        execute_plan(
            "Acme Corp",
            plan
        )
    )

    brief = (
        generate_meeting_brief(
            gathered_info
        )
    )

    add_message(
        "assistant",
        brief
    )

    print(
        "\nMeeting Brief"
    )

    print(
        "=" * 50
    )

    print(
        brief
    )


You:  prepare me for my meeting with acme crop



Agent Plan:
['client_profile', 'meeting_notes', 'memory', 'action_items']

Meeting Brief

    MEETING BRIEF

    CLIENT PROFILE
    --------------------------------------------------
    Acme Corp is a manufacturing company.

Primary contact:
John Smith

Industry:
Industrial Equipment

Current contract value:
$200,000

Interested in predictive maintenance solutions.


    PREVIOUS MEETING NOTES
    --------------------------------------------------
    Meeting Date: May 10

Discussed deployment timeline.

Client requested faster onboarding.

Client expressed interest in analytics dashboard.

Next review scheduled in June.


    LONG TERM MEMORY
    --------------------------------------------------
    {'priority': 'High', 'last_meeting': '2026-05-10', 'relationship_status': 'Active Client', 'notes': 'Interested in predictive maintenance solutions.'}

    OPEN ACTION ITEMS
    --------------------------------------------------
    1. Share analytics dashboard demo.

2. Send pricing pr


You:  exit


Agent: Goodbye!
